# Pipeline API 활용

> 토크나이저 → 모델 → 후처리를 **한 줄로** 끝내는 고수준 API

In [1]:
from transformers import pipeline
import torch

c:\Users\jskim\anaconda3\envs\distillation\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---
## 1. 텍스트 생성 (text-generation)

Decoder-only 모델 (GPT 계열) 사용

In [2]:
# === 텍스트 생성 Pipeline ===
# 03에서 tokenizer + model + generate()를 직접 했던 것을
# pipeline 한 줄로 대체
generator = pipeline(
    'text-generation',     # 태스크 지정
    model='gpt2',          # 모델 이름 (Hub에서 자동 다운로드)
    device=-1,             # -1=CPU, 0=첫번째 GPU
)

# 생성
results = generator(
    "The future of artificial intelligence",
    max_new_tokens=50,
    do_sample=True,
    temperature=0.7,
    num_return_sequences=2,  # 2개 결과 생성
)

print("텍스트 생성 결과:")
for i, r in enumerate(results):
    print(f"\n  [{i+1}] {r['generated_text'][:120]}...")

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 1097.89it/s, Materializing param=transformer.wte.weight]             
GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature', 'num_return_sequences', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/mai

텍스트 생성 결과:

  [1] The future of artificial intelligence in the workplace will be determined by how much the companies can afford."

In the...

  [2] The future of artificial intelligence will look bleak.

The next big challenge is how to keep up. The next big challenge...


---
## 2. 텍스트 분류 (text-classification)

Encoder-only 모델 (BERT 계열) 사용 - 감정 분석 등

In [3]:
# === 감정 분석 Pipeline ===
# BERT 계열 모델이 양방향 Attention으로 입력 전체를 파악 -> 분류
classifier = pipeline(
    'text-classification',
    model='distilbert-base-uncased-finetuned-sst-2-english',  # 감정분석 파인튰닝된 모델
    device=-1,
)

texts = [
    "I love this movie, it's fantastic!",
    "This is the worst experience ever.",
    "The weather is okay today.",
]

print("감정 분석 결과:")
results = classifier(texts)
for text, result in zip(texts, results):
    print(f"  '{text}'")
    print(f"    -> {result['label']} ({result['score']:.4f})\n")

print("-> Encoder-only(BERT) 모델: 양방향으로 전체 문맥 파악 -> 분류")
print("   Phase 1에서 배운 GPT(Decoder-only)와 반대 방향")

c:\Users\jskim\anaconda3\envs\distillation\lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jskim\.cache\huggingface\hub\models--distilbert-base-uncased-finetuned-sst-2-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 104/104 [00:00<00:00, 1093.84it/s, Materializ

감정 분석 결과:
  'I love this movie, it's fantastic!'
    -> POSITIVE (0.9999)

  'This is the worst experience ever.'
    -> NEGATIVE (0.9998)

  'The weather is okay today.'
    -> POSITIVE (0.9998)

-> Encoder-only(BERT) 모델: 양방향으로 전체 문맥 파악 -> 분류
   Phase 1에서 배운 GPT(Decoder-only)와 반대 방향


---
## 3. 질의응답 (question-answering)

주어진 문맥에서 답을 **추출**하는 태스크

In [4]:
# === 추출형 QA Pipeline ===
# 컨텍스트에서 답이 될 부분의 "\uc704\uce58"를 찾아내는 방식
# RAG의 Reader 부분과 직접 연결되는 개념
qa = pipeline(
    'question-answering',
    model='distilbert-base-cased-distilled-squad',
    device=-1,
)

context = """
Transformers were introduced in the paper "Attention Is All You Need" by Vaswani et al. in 2017.
The architecture replaced recurrent layers with self-attention mechanisms.
GPT-2, released by OpenAI in 2019, demonstrated that language models can generate coherent text.
BERT, developed by Google, uses bidirectional attention for understanding tasks.
"""

questions = [
    "When were Transformers introduced?",
    "Who released GPT-2?",
    "What does BERT use for understanding tasks?",
]

print("질의응답 (Extractive QA):")
print("=" * 50)
for q in questions:
    result = qa(question=q, context=context)
    print(f"  Q: {q}")
    print(f"  A: {result['answer']} (confidence: {result['score']:.4f})")
    print()

print("-> 컨텍스트 안에서 답을 '\ucd94\ucd9c'. RAG의 Reader 구성요소와 동일 원리")

c:\Users\jskim\anaconda3\envs\distillation\lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jskim\.cache\huggingface\hub\models--distilbert-base-cased-distilled-squad. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 102/102 [00:00<00:00, 1090.63it/s, Materializing param=

질의응답 (Extractive QA):
  Q: When were Transformers introduced?
  A: 2017 (confidence: 0.9697)

  Q: Who released GPT-2?
  A: OpenAI (confidence: 0.9963)

  Q: What does BERT use for understanding tasks?
  A: bidirectional attention (confidence: 0.9895)

-> 컨텍스트 안에서 답을 '추출'. RAG의 Reader 구성요소와 동일 원리


---
## 4. 요약 (summarization)

Encoder-Decoder 모델 (T5, BART) 사용

In [7]:
# === 요약 Pipeline ===
# Encoder-Decoder 모델: 입력 전체를 Encoder로 이해 -> Decoder로 요약 생성
# Phase 1에서 배운 Encoder/Decoder 구조의 실제 활용
summarizer = pipeline(
    'text-generation',
    model='sshleifer/distilbart-cnn-12-6',  # BART 기반 요약 모델
    device=-1,
)

article = """
Artificial intelligence has made remarkable progress in recent years, particularly
in the field of natural language processing. Large language models like GPT-4 and
Claude can now generate human-quality text, translate languages, and even write code.
These models are based on the Transformer architecture, which uses self-attention
mechanisms to process input sequences in parallel. The key innovation is the ability
to capture long-range dependencies in text without the limitations of recurrent
neural networks. Fine-tuning these models on specific tasks has become a standard
practice, with techniques like LoRA making it possible to adapt large models with
minimal computational resources.
"""

result = summarizer(article, max_length=60, min_length=20)

print("요약 결과:")
print(f"  원문 길이: {len(article.split())} 단어")
print(f"  요약: {result[0]['summary_text']}")
print(f"  요약 길이: {len(result[0]['summary_text'].split())} 단어")
print(f"\n-> Encoder-Decoder 모델(BART): 입력 이해(Encoder) + 요약 생성(Decoder)")

Please make sure the generation config includes `forced_bos_token_id=0`. 
Loading weights: 100%|██████████| 160/160 [00:00<00:00, 1332.77it/s, Materializing param=model.decoder.layers.5.self_attn_layer_norm.weight]   
BartForCausalLM LOAD REPORT from: sshleifer/distilbart-cnn-12-6
Key                                                       | Status     |  | 
----------------------------------------------------------+------------+--+-
model.encoder.layers.{0...11}.fc1.bias                    | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn.q_proj.bias       | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn.out_proj.weight   | UNEXPECTED |  | 
model.encoder.layernorm_embedding.weight                  | UNEXPECTED |  | 
model.encoder.layers.{0...11}.fc2.bias                    | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn.k_proj.weight     | UNEXPECTED |  | 
model.encoder.layers.{0...11}.final_layer_norm.bias       | UNEXPECTED |  | 
model.encoder.layers.{0..

요약 결과:
  원문 길이: 98 단어


KeyError: 'summary_text'

In [8]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained('sshleifer/distilbart-cnn-12-6')
model = AutoModelForSeq2SeqLM.from_pretrained('sshleifer/distilbart-cnn-12-6')

inputs = tokenizer(article, return_tensors='pt', max_length=1024, truncation=True)
outputs = model.generate(**inputs, max_length=60, min_length=20)
summary = tokenizer.decode(outputs[0], skip_special_tokens=True)

Loading weights: 100%|██████████| 358/358 [00:00<00:00, 1218.77it/s, Materializing param=model.shared.weight]                                  
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [9]:
summary

' Large language models like GPT-4 and Claude can generate human-quality text, translate languages, and even write code . These models are based on the Transformer architecture, which uses self-attention mechanisms to process input sequences in parallel .'

---
## 5. Feature Extraction (임베딩 추출)

RAG에서 문서를 벡터로 변환할 때 사용하는 그 원리

In [10]:
# === Feature Extraction: 텍스트 -> 벡터 ===
# RAG에서 문서를 임베딩할 때 내부적으로 이 과정이 일어남
extractor = pipeline(
    'feature-extraction',
    model='distilbert-base-uncased',
    device=-1,
)

texts = [
    "Machine learning is a subset of AI.",
    "Deep learning uses neural networks.",
]

# 각 텍스트를 벡터로 변환
for text in texts:
    features = extractor(text)
    # features: [batch, seq_len, hidden_dim]
    import numpy as np
    vec = np.array(features[0])  # [seq_len, 768]
    # 문장 임베딩: 모든 토큰의 평균 (mean pooling)
    sentence_vec = vec.mean(axis=0)  # [768]
    print(f"'{text}'")
    print(f"  토큰 수: {vec.shape[0]}, hidden_dim: {vec.shape[1]}")
    print(f"  문장 벡터: [{sentence_vec[0]:.4f}, {sentence_vec[1]:.4f}, ... ] (768차원)\n")

# 유사도 계산 (코사인 유사도)
vecs = []
for text in texts:
    features = extractor(text)
    vec = np.array(features[0]).mean(axis=0)
    vecs.append(vec)

cosine_sim = np.dot(vecs[0], vecs[1]) / (np.linalg.norm(vecs[0]) * np.linalg.norm(vecs[1]))
print(f"두 문장의 코사인 유사도: {cosine_sim:.4f}")
print(f"\n-> 이것이 RAG의 벡터 검색 원리")
print(f"   질문 벡터와 문서 벡터의 코사인 유사도로 관련 문서를 찾음")

c:\Users\jskim\anaconda3\envs\distillation\lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jskim\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1100.10it/s, Materializing param=transformer.la

'Machine learning is a subset of AI.'
  토큰 수: 10, hidden_dim: 768
  문장 벡터: [-0.0490, 0.0395, ... ] (768차원)

'Deep learning uses neural networks.'
  토큰 수: 8, hidden_dim: 768
  문장 벡터: [0.0378, 0.0215, ... ] (768차원)

두 문장의 코사인 유사도: 0.8760

-> 이것이 RAG의 벡터 검색 원리
   질문 벡터와 문서 벡터의 코사인 유사도로 관련 문서를 찾음


---
## 6. Pipeline vs 직접 코드: 언제 뭘 쓰나

In [11]:
# === 같은 결과를 Pipeline vs 직접 코드로 비교 ===
from transformers import AutoTokenizer, AutoModelForCausalLM

prompt = "The meaning of life is"

# 1) Pipeline (한 줄)
gen_pipe = pipeline('text-generation', model='gpt2', device=-1)
pipe_result = gen_pipe(prompt, max_new_tokens=30, do_sample=False)
print("Pipeline:")
print(f"  {pipe_result[0]['generated_text'][:100]}")

print()

# 2) 직접 코드 (세밀한 제어)
tokenizer = AutoTokenizer.from_pretrained('gpt2')
model = AutoModelForCausalLM.from_pretrained('gpt2')
inputs = tokenizer(prompt, return_tensors='pt')
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=30, do_sample=False)
direct_result = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("직접 코드:")
print(f"  {direct_result[:100]}")

print(f"\n동일 결과: {pipe_result[0]['generated_text'][:80] == direct_result[:80]}")

# 메모리 정리
del model, gen_pipe
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 1086.86it/s, Materializing param=transformer.wte.weight]             
GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer t

Pipeline:
  The meaning of life is not the same as the meaning of death.

The meaning of life is not the same as



Loading weights: 100%|██████████| 148/148 [00:00<00:00, 1081.75it/s, Materializing param=transformer.wte.weight]             
GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


직접 코드:
  The meaning of life is not the same as the meaning of death.

The meaning of life is not the same as

동일 결과: True


In [12]:
# === 상황별 선택 가이드 ===
guide = [
    ("모델 성능 빠르게 테스트", "Pipeline", "한 줄로 끝"),
    ("여러 모델 비교 실험", "Pipeline", "모델 이름만 바꿔서 테스트"),
    ("샘플링 파라미터 세밀 조절", "직접 코드", "logits 접근, 커스텀 로직"),
    ("Chat template 적용 대화", "직접 코드", "apply_chat_template() 사용"),
    ("배치 처리 / 프로덕션", "직접 코드 + vLLM", "성능 최적화 필수"),
    ("RAG 임베딩 생성", "Pipeline 또는 SentenceTransformers", "간편함 우선"),
]

h1, h2, h3 = "상황", "선택", "이유"
print(f"{h1:<30} {h2:<20} {h3}")
print("=" * 70)
for situation, choice, reason in guide:
    print(f"  {situation:<28} {choice:<18} {reason}")

상황                             선택                   이유
  모델 성능 빠르게 테스트                Pipeline           한 줄로 끝
  여러 모델 비교 실험                  Pipeline           모델 이름만 바꿔서 테스트
  샘플링 파라미터 세밀 조절               직접 코드              logits 접근, 커스텀 로직
  Chat template 적용 대화          직접 코드              apply_chat_template() 사용
  배치 처리 / 프로덕션                 직접 코드 + vLLM       성능 최적화 필수
  RAG 임베딩 생성                   Pipeline 또는 SentenceTransformers 간편함 우선


---
## 7. 모델 타입별 태스크 정리

In [13]:
# === 모델 타입별 적합한 태스크 정리 ===
# 이 대응 관계를 알면 "어떤 태스크에 어떤 모델"을 쓸지 바로 판단 가능

model_task_map = {
    "Encoder-only (BERT 계열)": {
        "특징": "양방향 Attention, Mask 없음",
        "태스크": ["분류", "감정분석", "NER", "추출형 QA", "임베딩/유사도"],
        "대표 모델": ["BERT", "RoBERTa", "DistilBERT"],
    },
    "Decoder-only (GPT 계열)": {
        "특징": "Causal Mask, 단방향 (Phase 1에서 구현)",
        "태스크": ["텍스트 생성", "ChatBot", "코드 생성", "Instruction Following"],
        "대표 모델": ["GPT-2/3/4", "Llama", "Mistral", "Qwen"],
    },
    "Encoder-Decoder (T5 계열)": {
        "특징": "Encoder(이해) + Cross-Attention + Decoder(생성)",
        "태스크": ["번역", "요약", "생성형 QA"],
        "대표 모델": ["T5", "BART", "mBART"],
    },
}

print("모델 타입별 적합 태스크:")
print("=" * 60)
for model_type, info in model_task_map.items():
    print(f"\n{model_type}")
    print(f"  특징: {info['특징']}")
    print(f"  태스크: {', '.join(info['태스크'])}")
    print(f"  대표: {', '.join(info['대표 모델'])}")

print("\n\ud575\uc2ec: \ud30c\uc778\ud2b0\ub2dd \ub300\uc0c1\uc740 \ub300\ubd80\ubd84 Decoder-only (GPT/Llama \uacc4\uc5f4)")
print("  -> Phase 3~4\uc5d0\uc11c Llama/Qwen \ubaa8\ub378\uc744 LoRA\ub85c \ud30c\uc778\ud2b0\ub2dd \uc608\uc815")

모델 타입별 적합 태스크:

Encoder-only (BERT 계열)
  특징: 양방향 Attention, Mask 없음
  태스크: 분류, 감정분석, NER, 추출형 QA, 임베딩/유사도
  대표: BERT, RoBERTa, DistilBERT

Decoder-only (GPT 계열)
  특징: Causal Mask, 단방향 (Phase 1에서 구현)
  태스크: 텍스트 생성, ChatBot, 코드 생성, Instruction Following
  대표: GPT-2/3/4, Llama, Mistral, Qwen

Encoder-Decoder (T5 계열)
  특징: Encoder(이해) + Cross-Attention + Decoder(생성)
  태스크: 번역, 요약, 생성형 QA
  대표: T5, BART, mBART

핵심: 파인튰닝 대상은 대부분 Decoder-only (GPT/Llama 계열)
  -> Phase 3~4에서 Llama/Qwen 모델을 LoRA로 파인튰닝 예정


---
## Phase 2 전체 정리

### 학습 흐름 요약

| # | 내용 | Phase 1 연결 |
|---|------|-------------|
| 01 | Auto 클래스, from_pretrained, 3가지 구조 | GPT 클래스 → AutoModelForCausalLM |
| 02 | Special tokens, Padding, Chat Template | BasicTokenizer → AutoTokenizer |
| 03 | 정밀도, device_map, generate() 샘플링 | softmax(logits/temp) → generate() |
| 04 | Datasets, map/filter, 데이터 포맷팅 | (새로운 내용) |
| 05 | Pipeline API, 모델타입별 태스크 | 전체 연결 |

### 다음 단계: Phase 3 (파인튜닝 데이터 준비)
- Instruction 데이터 형식 심화
- 데이터 품질 관리
- 합성 데이터 생성
- Phase 4에서 LoRA/QLoRA로 실제 파인튜닝